# Кольцевые коммуникации

Реализуйте три операции на CPU с Gloo. 
- `rank` — номер процесса,
- `world_size` — число процессов.
- Входные тензоры — одномерные CPU-тензоры одинакового размера на всех процессах. Их менять нельзя

In [2]:
%%writefile collectives.py
import torch
import torch.distributed as dist


def ring_reduce_scatter(tensor):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    if tensor.numel() % world_size != 0:
        raise ValueError("tensor size must be divisible by world_size")
    chunks = [chunk.clone() for chunk in tensor.chunk(world_size)]

    next_rank = (rank + 1) % world_size
    previous_rank = (rank - 1) % world_size
    for step in range(world_size - 1):
        send_index = (rank - step - 1) % world_size
        receive_index = (rank - step - 2) % world_size

        receive_buffer = torch.empty_like(chunks[receive_index])
        requests = dist.batch_isend_irecv(
            [
                dist.P2POp(
                    dist.isend,
                    chunks[send_index],
                    next_rank,
                ),
                dist.P2POp(
                    dist.irecv,
                    receive_buffer,
                    previous_rank,
                ),
            ]
        )

        for request in requests:
            request.wait()

        chunks[receive_index].add_(receive_buffer)
    return chunks[rank]


def ring_all_gather(shard):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    chunks = [torch.empty_like(shard) for _ in range(world_size)]
    chunks[rank].copy_(shard)
    next_rank = (rank + 1) % world_size
    previous_rank = (rank - 1) % world_size

    for step in range(world_size - 1):
        send_index = (rank - step) % world_size
        receive_index = (rank - step - 1) % world_size

        requests = dist.batch_isend_irecv(
            [
                dist.P2POp(
                    dist.isend,
                    chunks[send_index],
                    next_rank,
                ),
                dist.P2POp(
                    dist.irecv,
                    chunks[receive_index],
                    previous_rank,
                ),
            ]
        )

        for request in requests:
            request.wait()
    return torch.cat(chunks, dim=0)


def ring_all_reduce(tensor):
    reduced_shard = ring_reduce_scatter(tensor)
    return ring_all_gather(reduced_shard)


Overwriting collectives.py


## Сравнение с PyTorch

Сравним нашу реализацию с готовыми функциями торча `dist.reduce_scatter_tensor`, `dist.all_gather`
и `dist.all_reduce` на нескольких размерах тензоров. 

In [3]:
%%writefile check.py
from datetime import timedelta

import torch
import torch.distributed as dist

from collectives import ring_all_gather, ring_all_reduce, ring_reduce_scatter


torch.set_num_threads(1)
dist.init_process_group("gloo", timeout=timedelta(seconds=30))
try:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    generator = torch.Generator().manual_seed(42 + rank)

    for chunk_size in (1, 7, 31):
        values = torch.randn(world_size * chunk_size, generator=generator, dtype=torch.float64)
        original = values.clone()
        expected_shard = torch.empty(chunk_size, dtype=values.dtype)
        dist.reduce_scatter_tensor(expected_shard, values.clone(), op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_reduce_scatter(values), expected_shard)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

        shard = torch.randn(chunk_size, generator=generator, dtype=torch.float64)
        original_shard = shard.clone()
        expected_parts = [torch.empty_like(shard) for _ in range(world_size)]
        dist.all_gather(expected_parts, shard)
        torch.testing.assert_close(ring_all_gather(shard), torch.cat(expected_parts))
        torch.testing.assert_close(shard, original_shard, rtol=0, atol=0)

        expected_sum = values.clone()
        dist.all_reduce(expected_sum, op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_all_reduce(values), expected_sum)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

    if rank == 0:
        print(f"OK: все три операции прошли проверки на {world_size} процессах.")
finally:
    dist.destroy_process_group()

Writing check.py


In [1]:
import sys

!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=2 check.py

W1013 10:42:42.766000 67926 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W1013 10:42:43.216630000 socket.cpp:764] [c10d] The IPv6 network addresses of (MSK-G957H0G32G, 55005) cannot be retrieved (gai error: 8 - nodename nor servname provided, or not known).
[W1013 10:42:43.791263000 socket.cpp:764] [c10d] The IPv6 network addresses of (MSK-G957H0G32G, 55005) cannot be retrieved (gai error: 8 - nodename nor servname provided, or not known).
[W1013 10:42:44.292359000 socket.cpp:764] [c10d] The IPv6 network addresses of (MSK-G957H0G32G, 55005) cannot be retrieved (gai error: 8 - nodename nor servname provided, or not known).
[W1013

In [ ]:
!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=4 check.py

/Users/dbaymurzin/.venvs/evolutionrl-aviflow/bin/python: Error while finding module specification for 'torch.distributed.run' (ModuleNotFoundError: No module named 'torch')
